### Strathmore Institute of Mathematical Sciences | Master of Science in Data Science and Analytics
### DSA 8401 Applied Machine Learning | Capstone
#### 084045


#### Capstone Project: An End-to-End Behavioural Early-Warning Engine for Bank Account Dormancy and Disengagement Risk

**Dataset:** ai-lab/MBD-mini  
**Source:** Hugging Face 

### Imports

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.compose  import ColumnTransformer

from sklearn.preprocessing import (RobustScaler, OneHotEncoder)

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

print(f'Imports completed')

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.11.0' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import (
/opt/anaconda3/lib/python3.13/site-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


Imports completed


In [3]:
# install hugging face package
%pip install -U datasets huggingface_hub pandas

Note: you may need to restart the kernel to use updated packages.


### Data Download

**Source:** [MBD-mini](https://huggingface.co/datasets/ai-lab/MBD-mini) (Sber AI Lab,
licence CC BY 4.0), a 10% client sample of the Multimodal Banking Dataset
([Mollaev et al., 2024](https://arxiv.org/abs/2409.17587)).

The repository holds four archives. Download detail, client_split, targets and ptls files:

| Archive | Contents|
|---|---|
| `detail.tar.gz` | Raw event tables: transactions, geo-activity, support dialogues|
| `client_split.tar.gz` | Client-to-fold assignment|
| `targets.tar.gz` | The authors' monthly product-purchase targets|
| `ptls.tar.gz` | Pre-aggregated sequences for one specific library|

In [4]:
# download MBD-mini file; target.tar.gz 
from huggingface_hub import hf_hub_download

targets_path = hf_hub_download(
    repo_id="ai-lab/MBD-mini",
    filename="targets.tar.gz",
    repo_type="dataset"
)

print(targets_path)

/Users/tintin/.cache/huggingface/hub/datasets--ai-lab--MBD-mini/snapshots/4b0d5b4c0b014e02af6c2e6b7b27851e6fb0d5c3/targets.tar.gz


In [5]:
# download  detail.tar.gz file
from huggingface_hub import hf_hub_download

detail_path = hf_hub_download(
    repo_id="ai-lab/MBD-mini",
    filename="detail.tar.gz",
    repo_type="dataset"
)

print(detail_path)

/Users/tintin/.cache/huggingface/hub/datasets--ai-lab--MBD-mini/snapshots/4b0d5b4c0b014e02af6c2e6b7b27851e6fb0d5c3/detail.tar.gz


In [6]:
# download  client_split.tar.gz file
from huggingface_hub import hf_hub_download

client_split_path = hf_hub_download(
    repo_id="ai-lab/MBD-mini",
    filename="client_split.tar.gz",
    repo_type="dataset"
)

print(client_split_path)

/Users/tintin/.cache/huggingface/hub/datasets--ai-lab--MBD-mini/snapshots/4b0d5b4c0b014e02af6c2e6b7b27851e6fb0d5c3/client_split.tar.gz


In [7]:
# download  ptls.tar.gz file
from huggingface_hub import hf_hub_download

ptls_path = hf_hub_download(
    repo_id="ai-lab/MBD-mini",
    filename="ptls.tar.gz",
    repo_type="dataset"
)

print(ptls_path)

/Users/tintin/.cache/huggingface/hub/datasets--ai-lab--MBD-mini/snapshots/4b0d5b4c0b014e02af6c2e6b7b27851e6fb0d5c3/ptls.tar.gz


In [8]:
import os

dataset_path = os.path.expanduser("~/Documents/School//School/MSc. DSA 4/DSA 8401 Applied Machine Learning/Capstone Project")

os.makedirs(dataset_path, exist_ok=True)

print(dataset_path)

/Users/tintin/Documents/School//School/MSc. DSA 4/DSA 8401 Applied Machine Learning/Capstone Project


In [9]:
dataset_path = (
    "/Users/tintin/.cache/huggingface/hub/"
    "datasets--ai-lab--MBD-mini/snapshots/"
    "4b0d5b4c0b014e02af6c2e6b7b27851e6fb0d5c3"
)

print(os.path.exists(dataset_path))

True


In [10]:
files = os.listdir(dataset_path)

for file in files:
    if file.endswith(".tar.gz"):
        file_path = os.path.join(dataset_path, file)

        size = os.path.getsize(file_path) / (1024**3)

        print(f"{file}: {size:.2f} GB")

targets.tar.gz: 0.01 GB
ptls.tar.gz: 1.30 GB
detail.tar.gz: 1.83 GB
client_split.tar.gz: 0.00 GB


In [11]:
# extracting all four archives
import tarfile

archives = [
    "targets.tar.gz",
    "client_split.tar.gz",
    "detail.tar.gz",
    "ptls.tar.gz"
]

for archive in archives:

    archive_path = os.path.join(dataset_path, archive)

    folder_name = archive.replace(".tar.gz", "")
    extract_path = os.path.join(dataset_path, folder_name)

    os.makedirs(extract_path, exist_ok=True)

    with tarfile.open(archive_path, "r:gz") as tar:
        tar.extractall(path=extract_path, filter="data")

    print(f"Successfully extracted: {archive}")

Successfully extracted: targets.tar.gz
Successfully extracted: client_split.tar.gz
Successfully extracted: detail.tar.gz
Successfully extracted: ptls.tar.gz


In [12]:
# inspection of extracted files
for folder in ["targets", "client_split", "detail", "ptls"]:

    folder_path = os.path.join(dataset_path, folder)

    print(f"\n{'='*50}")
    print(f"FOLDER: {folder}")
    print('='*50)

    for root, dirs, files in os.walk(folder_path):
        for file in files:
            file_path = os.path.join(root, file)

            size = os.path.getsize(file_path) / (1024**2)

            print(f"{file} | {size:.2f} MB")


FOLDER: targets
_SUCCESS | 0.00 MB
part-00083-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet | 0.04 MB
part-00036-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet | 0.03 MB
part-00042-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet | 0.02 MB
part-00063-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet | 0.03 MB
part-00017-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet | 0.03 MB
part-00068-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet | 0.02 MB
part-00021-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet | 0.03 MB
part-00055-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet | 0.02 MB
part-00074-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet | 0.02 MB
part-00000-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet | 0.02 MB
part-00049-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet | 0.03 MB
part-00045-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet | 0.02 MB
part-00031-44ca8b70-9d42-48f

In [14]:
# locate parquet files
from pathlib import Path

targets_dir = Path(dataset_path) / "targets"

parquet_files = list(targets_dir.rglob("*.parquet"))

print("Number of Parquet files:", len(parquet_files))

# Display the first 5 file paths
for file in parquet_files[:5]:
    print(file.relative_to(targets_dir))

Number of Parquet files: 435
targets/fold=0/part-00083-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet
targets/fold=0/part-00036-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet
targets/fold=0/part-00042-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet
targets/fold=0/part-00063-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet
targets/fold=0/part-00017-44ca8b70-9d42-48f7-9dec-0a7a012af308.c000.snappy.parquet


In [13]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


In [15]:
import pyarrow.dataset as ds

In [16]:
EXTRACTED = Path(dataset_path)

TABLES = {
    "client_split": EXTRACTED / "client_split" / "client_split",
    "trx": EXTRACTED / "detail" / "detail" / "trx",
    "geo": EXTRACTED / "detail" / "detail" / "geo",
    "dialog": EXTRACTED / "detail" / "detail" / "dialog",
    "targets": EXTRACTED / "targets" / "targets",
}

In [17]:
for name, path in TABLES.items():
    print(f"{name}: {'Found' if path.exists() else 'NOT FOUND'}")

client_split: Found
trx: Found
geo: Found
dialog: Found
targets: Found


In [18]:
datasets = {
    name: ds.dataset(path, format="parquet", partitioning="hive")
    for name, path in TABLES.items()
}

### Inventory: size, rows and columns

In [19]:
inventory = []

for name, dset in datasets.items():
    files = list(TABLES[name].rglob("*.parquet"))

    inventory.append({
        "table": name,
        "files": len(files),
        "size_MB": sum(f.stat().st_size for f in files) / 1e6,
        "rows": dset.count_rows(),
        "columns": len(dset.schema),
    })

inventory = pd.DataFrame(inventory).set_index("table")

inventory.style.format({
    "size_MB": "{:,.1f}",
    "rows": "{:,}",
})

,files,size_MB,rows,columns
table,,,,
client_split,1000,7.5,"100,224",2
trx,400,677.3,"38,764,930",15
geo,460,832.5,"66,295,724",6
dialog,415,673.9,"217,205",4
targets,435,11.5,"1,202,688",9


In [ ]:
# inspect tables schema; data types
for name, dset in datasets.items():
    print(f"\n{'=' * 60}")
    print(f"TABLE: {name}")
    print(f"{'=' * 60}")
    print(dset.schema)


TABLE: client_split
client_id: string
fold: int32
-- schema metadata --
org.apache.spark.version: '3.2.1'
org.apache.spark.sql.parquet.row.metadata: '{"type":"struct","fields":[{"' + 66

TABLE: trx
client_id: string
event_time: timestamp[ns]
amount: float
event_type: int32
event_subtype: int32
currency: int32
src_type11: int32
src_type12: int32
dst_type11: int32
dst_type12: int32
src_type21: int32
src_type22: int32
src_type31: int32
src_type32: int32
fold: int32
-- schema metadata --
org.apache.spark.version: '3.2.1'
org.apache.spark.sql.parquet.row.metadata: '{"type":"struct","fields":[{"' + 960

TABLE: geo
client_id: string
event_time: timestamp[ns]
geohash_4: int32
geohash_5: int32
geohash_6: int32
fold: int32
-- schema metadata --
org.apache.spark.version: '3.2.1'
org.apache.spark.sql.parquet.row.metadata: '{"type":"struct","fields":[{"' + 341

TABLE: dialog
client_id: string
event_time: timestamp[ns]
embedding: list<element: float>
  child 0, element: float
fold: int32
-- schema 

In [ ]:
# read data
for name, dset in datasets.items():
    print(f"\nTABLE: {name}")
    display(dset.head(5).to_pandas())


TABLE: client_split


,client_id,fold
0,091d957b9e571dc1ddb66017f32202d3a4f48583b6399a...,0
1,1171cec680efbb4fd268a9ade0e94512c7a6f1e08a98a6...,0
2,127042b9a315782dfba125603f5173144f2b1e9b86b0bd...,0
3,14a3fab5925a8c9271ab9e16c25fedb47122095476a741...,0
4,2421da099471e3c4cca875717d19c6853a80930a01fad1...,0



TABLE: trx


,client_id,event_time,amount,event_type,event_subtype,currency,src_type11,src_type12,dst_type11,dst_type12,src_type21,src_type22,src_type31,src_type32,fold
0,01a1aca1531e0d86b605759fedd36e1855f41a8a6a49b4...,2022-10-26 13:58:59.938408,1.021543e+06,16,18,11,22,47,780,21619,21490,83,364,58,0
1,01a1aca1531e0d86b605759fedd36e1855f41a8a6a49b4...,2021-02-09 07:43:20.639579,3.007656e+05,1,12,11,4,958,306,26766,21490,83,364,58,0
2,01a1aca1531e0d86b605759fedd36e1855f41a8a6a49b4...,2022-10-22 09:12:06.614108,6.061549e+05,16,18,11,22,47,780,21619,21490,83,364,58,0
3,01a1aca1531e0d86b605759fedd36e1855f41a8a6a49b4...,2021-02-12 15:12:20.660481,8.032591e+04,1,12,11,4,958,306,26766,21490,83,364,58,0
4,01a1aca1531e0d86b605759fedd36e1855f41a8a6a49b4...,2022-10-08 06:52:28.877461,1.222866e+05,16,18,11,22,47,780,21619,21490,83,364,58,0



TABLE: geo


,client_id,event_time,geohash_4,geohash_5,geohash_6,fold
0,397d52b11ce72699a5482e5397f63a03af1636c85e9e61...,2022-03-25 13:39:51.569288,41342,406579,1018528,0
1,397d52b11ce72699a5482e5397f63a03af1636c85e9e61...,2022-06-11 07:21:32.505485,41342,406579,1018528,0
2,397d52b11ce72699a5482e5397f63a03af1636c85e9e61...,2022-08-08 21:25:07.461323,41342,406579,1018528,0
3,397d52b11ce72699a5482e5397f63a03af1636c85e9e61...,2022-07-23 11:32:03.433401,41342,406579,1018528,0
4,397d52b11ce72699a5482e5397f63a03af1636c85e9e61...,2022-05-28 08:18:19.843885,41342,406579,1018528,0



TABLE: dialog


,client_id,event_time,embedding,fold
0,5226901c78c655767ac2a075e6edca6b410624f0c9d2fd...,2021-06-18 05:49:23.962741,"[0.393141, -0.036111, 0.311134, -0.197847, -0....",0
1,5226901c78c655767ac2a075e6edca6b410624f0c9d2fd...,2022-06-29 07:39:31.618769,"[0.302206, -0.381503, 0.472719, -0.284991, -0....",0
2,5226901c78c655767ac2a075e6edca6b410624f0c9d2fd...,2022-06-24 06:06:59.691367,"[0.570911, -0.395088, 0.546858, -0.413399, -0....",0
3,5226901c78c655767ac2a075e6edca6b410624f0c9d2fd...,2021-05-12 12:34:32.616245,"[0.232727, -0.284487, 0.584149, -0.524622, -0....",0
4,5226901c78c655767ac2a075e6edca6b410624f0c9d2fd...,2021-10-25 05:25:17.810891,"[0.316968, -0.149733, 0.426037, -0.395491, -0....",0



TABLE: targets


,client_id,mon,target_1,target_2,target_3,target_4,trans_count,diff_trans_date,fold
0,00bd0ecf3d5a33aa8756097967d07797dca4c98de9b61c...,2022-02-28,0,0,0,0,10,0,0
1,00bd0ecf3d5a33aa8756097967d07797dca4c98de9b61c...,2022-03-31,0,0,0,0,29,0,0
2,00bd0ecf3d5a33aa8756097967d07797dca4c98de9b61c...,2022-04-30,0,0,0,0,51,0,0
3,00bd0ecf3d5a33aa8756097967d07797dca4c98de9b61c...,2022-05-31,0,0,0,0,52,4,0
4,00bd0ecf3d5a33aa8756097967d07797dca4c98de9b61c...,2022-06-30,0,0,0,0,52,35,0


In [22]:
def peek(name: str, n: int = 5) -> pd.DataFrame:
    df = datasets[name].head(n).to_pandas()
    if "embedding" in df:  # 768 floats per row; show the length only
        df["embedding"] = df["embedding"].map(lambda v: f"<{len(v)} floats>")
    return df


print(datasets["trx"].schema.remove_metadata())
peek("trx")

client_id: string
event_time: timestamp[ns]
amount: float
event_type: int32
event_subtype: int32
currency: int32
src_type11: int32
src_type12: int32
dst_type11: int32
dst_type12: int32
src_type21: int32
src_type22: int32
src_type31: int32
src_type32: int32
fold: int32


,client_id,event_time,amount,event_type,event_subtype,currency,src_type11,src_type12,dst_type11,dst_type12,src_type21,src_type22,src_type31,src_type32,fold
0,01a1aca1531e0d86b605759fedd36e1855f41a8a6a49b4...,2022-10-26 13:58:59.938408,1.021543e+06,16,18,11,22,47,780,21619,21490,83,364,58,0
1,01a1aca1531e0d86b605759fedd36e1855f41a8a6a49b4...,2021-02-09 07:43:20.639579,3.007656e+05,1,12,11,4,958,306,26766,21490,83,364,58,0
2,01a1aca1531e0d86b605759fedd36e1855f41a8a6a49b4...,2022-10-22 09:12:06.614108,6.061549e+05,16,18,11,22,47,780,21619,21490,83,364,58,0
3,01a1aca1531e0d86b605759fedd36e1855f41a8a6a49b4...,2021-02-12 15:12:20.660481,8.032591e+04,1,12,11,4,958,306,26766,21490,83,364,58,0
4,01a1aca1531e0d86b605759fedd36e1855f41a8a6a49b4...,2022-10-08 06:52:28.877461,1.222866e+05,16,18,11,22,47,780,21619,21490,83,364,58,0


In [23]:
print(datasets["geo"].schema.remove_metadata())
peek("geo")

client_id: string
event_time: timestamp[ns]
geohash_4: int32
geohash_5: int32
geohash_6: int32
fold: int32


,client_id,event_time,geohash_4,geohash_5,geohash_6,fold
0,397d52b11ce72699a5482e5397f63a03af1636c85e9e61...,2022-03-25 13:39:51.569288,41342,406579,1018528,0
1,397d52b11ce72699a5482e5397f63a03af1636c85e9e61...,2022-06-11 07:21:32.505485,41342,406579,1018528,0
2,397d52b11ce72699a5482e5397f63a03af1636c85e9e61...,2022-08-08 21:25:07.461323,41342,406579,1018528,0
3,397d52b11ce72699a5482e5397f63a03af1636c85e9e61...,2022-07-23 11:32:03.433401,41342,406579,1018528,0
4,397d52b11ce72699a5482e5397f63a03af1636c85e9e61...,2022-05-28 08:18:19.843885,41342,406579,1018528,0


In [24]:
print(datasets["dialog"].schema.remove_metadata())
peek("dialog")

client_id: string
event_time: timestamp[ns]
embedding: list<element: float>
  child 0, element: float
fold: int32


,client_id,event_time,embedding,fold
0,5226901c78c655767ac2a075e6edca6b410624f0c9d2fd...,2021-06-18 05:49:23.962741,<768 floats>,0
1,5226901c78c655767ac2a075e6edca6b410624f0c9d2fd...,2022-06-29 07:39:31.618769,<768 floats>,0
2,5226901c78c655767ac2a075e6edca6b410624f0c9d2fd...,2022-06-24 06:06:59.691367,<768 floats>,0
3,5226901c78c655767ac2a075e6edca6b410624f0c9d2fd...,2021-05-12 12:34:32.616245,<768 floats>,0
4,5226901c78c655767ac2a075e6edca6b410624f0c9d2fd...,2021-10-25 05:25:17.810891,<768 floats>,0


In [25]:
print(datasets["targets"].schema.remove_metadata())
peek("targets")

client_id: string
mon: string
target_1: int32
target_2: int32
target_3: int32
target_4: int32
trans_count: int32
diff_trans_date: int32
fold: int32


,client_id,mon,target_1,target_2,target_3,target_4,trans_count,diff_trans_date,fold
0,00bd0ecf3d5a33aa8756097967d07797dca4c98de9b61c...,2022-02-28,0,0,0,0,10,0,0
1,00bd0ecf3d5a33aa8756097967d07797dca4c98de9b61c...,2022-03-31,0,0,0,0,29,0,0
2,00bd0ecf3d5a33aa8756097967d07797dca4c98de9b61c...,2022-04-30,0,0,0,0,51,0,0
3,00bd0ecf3d5a33aa8756097967d07797dca4c98de9b61c...,2022-05-31,0,0,0,0,52,4,0
4,00bd0ecf3d5a33aa8756097967d07797dca4c98de9b61c...,2022-06-30,0,0,0,0,52,35,0


In [26]:
print(datasets["client_split"].schema.remove_metadata())
peek("client_split")

client_id: string
fold: int32


,client_id,fold
0,091d957b9e571dc1ddb66017f32202d3a4f48583b6399a...,0
1,1171cec680efbb4fd268a9ade0e94512c7a6f1e08a98a6...,0
2,127042b9a315782dfba125603f5173144f2b1e9b86b0bd...,0
3,14a3fab5925a8c9271ab9e16c25fedb47122095476a741...,0
4,2421da099471e3c4cca875717d19c6853a80930a01fad1...,0


### Time span of each table

In [28]:
import pyarrow.compute as pc

spans = []
for name in ["trx", "geo", "dialog"]:
    lo = hi = None
    for batch in datasets[name].to_batches(columns=["event_time"], batch_size=2_000_000):
        mm = pc.min_max(batch.column(0))
        lo = mm["min"].as_py() if lo is None else min(lo, mm["min"].as_py())
        hi = mm["max"].as_py() if hi is None else max(hi, mm["max"].as_py())
    spans.append({"table": name, "first_event": lo, "last_event": hi})

months = datasets["targets"].to_table(columns=["mon"]).column(0)
spans.append({"table": "targets (mon)", "first_event": pc.min(months).as_py(), "last_event": pc.max(months).as_py()})
pd.DataFrame(spans).set_index("table")

,first_event,last_event
table,,
trx,2020-12-31 21:00:02.169545,2022-12-31 20:59:58.460193
geo,2021-12-11 19:27:07.507106,2022-12-31 20:59:58.725788
dialog,2021-02-10 09:36:07.022271,2022-12-31 20:25:11.730797
targets (mon),2022-02-28,2023-01-31


- **Transactions span 24 months** (January 2021 to December 2022). The dataset card
  says 12 months, which describes the `targets` table, not the event tables.
- **Geo data starts only in December 2021**, so geo features can exist only for scoring
  dates in 2022.

### Which clients appear in which table?

In [29]:
def unique_clients(dset) -> set:
    # Batch by batch: geo alone has 66M rows, and loading its client_id column
    # at once needs more memory than a typical laptop has.
    seen = set()
    for batch in dset.to_batches(columns=["client_id"], batch_size=1_000_000):
        seen.update(pc.unique(batch.column(0)).to_pylist())
    return seen


clients = {name: unique_clients(dset) for name, dset in datasets.items()}
all_clients = clients["client_split"]

coverage = pd.DataFrame({
    "clients": {n: len(c) for n, c in clients.items()},
    "share_of_all_clients": {n: len(c & all_clients) / len(all_clients) for n, c in clients.items()},
    "not_in_client_split": {n: len(c - all_clients) for n, c in clients.items()},
})
coverage.style.format({"clients": "{:,}", "share_of_all_clients": "{:.1%}", "not_in_client_split": "{:,}"})

,clients,share_of_all_clients,not_in_client_split
client_split,"100,224",100.0%,0
trx,"98,721",98.5%,0
geo,"72,573",72.4%,0
dialog,"46,320",46.2%,0
targets,"100,224",100.0%,0


In [30]:
no_trx = all_clients - clients["trx"]
print(f"Clients with no transactions at all: {len(no_trx):,}")
print(f"Clients with transactions but no geo: {len(clients['trx'] - clients['geo']):,}")
print(f"Clients with transactions but no dialogue: {len(clients['trx'] - clients['dialog']):,}")

Clients with no transactions at all: 1,503
Clients with transactions but no geo: 27,074
Clients with transactions but no dialogue: 52,715


### Summary

| | |
|---|---|
| Population | 100,224 corporate clients (`client_split`) |
| Core table | `trx`: 38.8M transactions, 24 months |
| Optional sources | `geo` (2022 only, ~73% of clients), `dialog` (~46% of clients) |
| Not used as a label | `targets` (product purchases, a different task) |

- Dormancy is not given; we will **derive it from `trx`** (no transaction for N days).
- Clients with no transactions can never "become" dormant, so they leave the population.
- Missing `geo` or `dialog` for a client is normal (a missing *modality*), not a data error.

### Data preparation and cleaning